## B_01 · Calidad y limpieza (issue #9)
Perfil de calidad de las 6 carreras reales. Nunca se muestran nombres ni dorsales: siempre se pasa por `quitar_identificadores()` o solo se muestran recuentos.

### Preparar

In [16]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from src.config import DIR_INTERIM
from src.proteccion import quitar_identificadores

### Cargar las 6 carreras tal cual vienen
Se leen los CSV de `datos/interim/` que genera `02_carreras.ipynb` (no están en el repo porque llevan nombres).

In [17]:
carreras = {}
for ruta in sorted(DIR_INTERIM.glob("carrera_0?.csv")):
    carreras[ruta.stem] = pd.read_csv(ruta, dtype=str, keep_default_na=False)

for nombre, df in carreras.items():
    print(nombre, df.shape)

carrera_01 (1489, 7)
carrera_02 (929, 7)
carrera_03 (795, 7)
carrera_04 (875, 7)
carrera_05 (659, 7)
carrera_06 (643, 7)


### Primer vistazo, siempre sin nombres

In [18]:
quitar_identificadores(carreras["carrera_01"]).head()

,posto,posto_sexo,posto_categoria,tempo
0,1,M-1,SNRM-1,00:20:30
1,2,M-2,SNRM-2,00:21:00
2,3,M-3,SUB23M-1,00:21:01
3,4,M-4,SNRM-3,00:21:19
4,5,M-5,MS1M-1,00:21:26


Perfil de calidad

In [19]:
from src.config import DIR_EXTERNOS
from src.limpieza import tiempo_a_segundos, marcar_tiempo, parsear_posto_sexo, parsear_posto_categoria

# Distancia de cada carrera (para el tiempo mínimo) y códigos de categoría válidos
distancias = pd.read_csv(DIR_EXTERNOS / "distancias.csv").set_index("carrera_id")["distancia_m"]
categorias_validas = set(pd.read_csv(DIR_EXTERNOS / "categorias.csv")["codigo_carrera"]) - {"?"}


def perfil(df, carrera_id):
    """Cuenta los problemas de calidad de una carrera. Solo devuelve números, nunca nombres."""
    # Separar las columnas mezcladas con las funciones de src/limpieza.py
    sexo_ps = df["posto_sexo"].map(lambda t: parsear_posto_sexo(t)[0])
    cat = df["posto_categoria"].map(parsear_posto_categoria)
    cat_codigo = cat.map(lambda t: t[0])
    cat_sexo = cat.map(lambda t: t[1])
    segundos = df["tempo"].map(tiempo_a_segundos)
    marca = segundos.map(lambda s: marcar_tiempo(s, distancias.get(carrera_id)))

    return {
        "carrera_id": carrera_id,
        "filas": len(df),
        # Completitud
        "vacios": int((df == "").sum().sum()),
        # Validez
        "posto_no_numerico": int((~df["posto"].str.fullmatch(r"\d+")).sum()),
        "posto_sexo_mal_formato": int(sexo_ps.isna().sum()),
        "posto_categoria_mal_formato": int((cat_codigo.isna() & (df["posto_categoria"] != "NO")).sum()),
        "categoria_NO": int((df["posto_categoria"] == "NO").sum()),
        "categoria_ESP": int((cat_codigo == "ESP").sum()),
        "categoria_desconocida": int((cat_codigo.notna() & ~cat_codigo.isin(categorias_validas)).sum()),
        # Consistencia: el sexo de M-3 debe coincidir con el de MS2M-1
        "sexo_no_coincide": int((cat_sexo.notna() & sexo_ps.notna() & (cat_sexo != sexo_ps)).sum()),
        # Completitud y exactitud de los tiempos
        "tempo_mal_formato": int((segundos.isna() & (df["tempo"] != "")).sum()),
        "tempo_faltante": int((marca == "faltante").sum()),
        "tempo_imposible": int((marca == "imposible").sum()),
        # Unicidad
        "posto_repetido": int(df["posto"].duplicated().sum()),
        "dorsal_repetido": int(df["dorsal"].duplicated().sum()),
        "filas_duplicadas": int(df.duplicated().sum()),
    }

### Perfil de las 6 carreras
Una columna por carrera y una fila por problema (`.T` gira la tabla para leerla mejor).

In [20]:
tabla_perfil = pd.DataFrame([perfil(df, nombre) for nombre, df in carreras.items()])
tabla_perfil.set_index("carrera_id").T

carrera_id,carrera_01,carrera_02,carrera_03,carrera_04,carrera_05,carrera_06
filas,1489,929,795,875,659,643
vacios,0,0,0,0,0,0
posto_no_numerico,0,0,0,0,0,0
posto_sexo_mal_formato,3,6,5,0,3,0
posto_categoria_mal_formato,3,6,5,0,3,0
categoria_NO,0,0,0,0,0,0
categoria_ESP,1,0,0,0,1,1
categoria_desconocida,0,0,0,0,0,0
sexo_no_coincide,0,0,0,0,0,0
tempo_mal_formato,0,0,0,0,0,0
